<a href="https://colab.research.google.com/github/genji970/LLM_pretraining/blob/main/pretraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers

# **data preparation**

In [ ]:
from dataclasses import dataclass
import string , random

@dataclass
class Data_Config:
  data_path : str = None
  # task of data / purpose of data
  data_kind : str = None
  # Number of data
  sample_num : int = None
  # How many data will be loaded in Dataset
  chunk : int = None
  name : str = None

  token_key : str = None
  token_id : int = None

  data_structure : tuple | None = ("question" , "reference" , "answer")

  # what is field()?

In [ ]:
from typing import Callable , Union
import os, timeit, psutil
from datasets import (
    Dataset,
    load_dataset,
    IterableDataset,
    concatenate_datasets,
    interleave_datasets,
)

class Data_Preprocessing:
  def __init__(self , answer_type:str='text',seed_value=42):

    if answer_type not in {"text" , "letter"}:
      raise ValueError("answer_type should be either text or letter!")

    self.answer_type = answer_type
    self.seed_value = seed_value

  def data_download(self, dataset_name):
    if dataset_name not in ['SuperGPQA' , 'kina']:
      raise ValueError("dataset_name you try to load is not in current dataset list")

    if dataset_name == "SuperGPQA":
      mem_before = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
      SuperGPQA = load_dataset("m-a-p/SuperGPQA",split="train")
      SuperGPQA = SuperGPQA.train_test_split(test_size=0.3, seed=self.seed_value)
      SuperGPQA_train = SuperGPQA['train']
      SuperGPQA_test = SuperGPQA['test']
      mem_after = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
      print(f"RAM memory used: {(mem_after - mem_before)} MB")
      return SuperGPQA_train, SuperGPQA_test

    if dataset_name == "kina":
      mem_before = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
      kina = load_dataset("2077AIDataFoundation/KINA",split="test")
      mem_after = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
      print(f"RAM memory used: {(mem_after - mem_before)} MB")
      return kina

  def divide_by_difficulty(self,dataset : Union[Dataset , IterableDataset],difficulty_key : str,difficulty : str):

    if difficulty_key not in dataset.column_names:
      pass

    return dataset.filter(
        lambda row: row[difficulty_key] == difficulty
    )
  def divide_by_dataset_name(self, dataset : Union[Dataset , IterableDataset], dataset_name_key : str,dataset_name : str):

    if dataset_name_key not in dataset.column_names:
      raise ValueError(f"there's no {dataset_name_key} in this dataset")

    return dataset.filter(
        lambda row: row[dataset_name_key] == dataset_name
    )

  def calculate_length(self, dataset):
    required_columns = ["question", "reference", "options", "answer"]

    if not all(
      column in dataset.column_names
      for column in required_columns
      ):
      raise ValueError("essential column name/names is/are not in the dataset column list")

    question_max_length = max(
        len(str(question))
        for question in dataset["question"]
    )

    reference_max_length = max(
        len(str(reference))
        for reference in dataset["reference"]
    )

    options_max_length = max(
        sum(len(str(option)) for option in options)
        for options in dataset["options"]
    )

    answer_max_length = max(
        len(str(answer))
        for answer in dataset["answer"]
    )

    length = (
        question_max_length
        + reference_max_length
        + options_max_length
        + answer_max_length
    )

    return length

  def build_mcqa_pretrain_prompt(self,row):
    options=row['options']

    if not options:
      raise ValueError("The options list is empty")

    if isinstance(options[0] , dict):
      options_text = "\n".join(
          f"{option['key']}. {option['answer'].strip()}"
          for option in options
      )
    else:
      options_text="\n".join(
          f"{chr(65 + index)}. {str(option).strip()}"
          for index, option in enumerate(options)
      )
    return {
        "prompt": (
            f"Question:\n"
            f"{row['question'].strip()}\n\n"
            f"The answer choices are as follows:\n"
            f"{options_text}\n\n"
            f"Select the correct answer."
            f"The correct answer letter is {row['answer_letter'].strip()}."
            f"And the correct answer is {row['answer'].strip()}."
        )
    }

  def build_mcqa_prompt(self,row):
    options = row["options"]
    if not options:
        raise ValueError("The options list is empty.")

    if isinstance(options[0], dict):
        options_text = "\n".join(
            f"{option['key']}. {option['answer'].strip()}"
            for option in options
        )
    else:
        options_text = "\n".join(
            f"{chr(65 + index)}. {str(option).strip()}"
            for index, option in enumerate(options)
        )

    return {
        "prompt": (
            f"Question:\n"
            f"{row['question'].strip()}\n\n"
            f"The answer choices are as follows:\n"
            f"{options_text}\n\n"
            f"Select the correct answer and respond with only its letter."
        )
    }

  def column_name_interpolating(self, dataset , dataset_name):
    """
    dataset.column == (id, question , reference , options, answer , answer_letter,difficulty)
    """

    unnecessary_columns_in_SuperGPQA = ['discipline', 'field', 'subfield', 'is_calculation']
    unnecessary_columns_in_kina=['discipline']

    if dataset_name == "SuperGPQA":
      dataset=dataset.rename_column(
          'uuid' , 'id'
      )

      dataset=dataset.remove_columns(
          unnecessary_columns_in_SuperGPQA
      )

      if 'reference' not in dataset.column_names:
        dataset=dataset.map(
            lambda row: {'reference' : ""}
        )

    if dataset_name == "kina":
      dataset=dataset.rename_columns(
          {'index' : 'id',
          'correct_answer' : 'answer'
          }
      )

      dataset=dataset.remove_columns(
          unnecessary_columns_in_kina
      )

      if 'reference' not in dataset.column_names:
        dataset=dataset.map(
            lambda row: {'reference' : ""}
        )

    return dataset

In [ ]:
data_process=Data_Preprocessing()

SuperGPQA_train, SuperGPQA_test=data_process.data_download('SuperGPQA')
kina=data_process.data_download('kina')

SuperGPQA_train=data_process.column_name_interpolating(SuperGPQA_train , "SuperGPQA")
SuperGPQA_test=data_process.column_name_interpolating(SuperGPQA_test , "SuperGPQA")
kina=data_process.column_name_interpolating(kina , "kina")

print(f"superGPQA_key : {SuperGPQA_train.column_names} , kina_key : {kina.column_names}")

SuperGPQA_train_max_length=data_process.calculate_length(SuperGPQA_train)
SuperGPQA_test_max_length=data_process.calculate_length(SuperGPQA_test)
kina_max_length=data_process.calculate_length(kina)

print(f"max length for SuperGPQA train : {SuperGPQA_train_max_length} , SuperGPQA test : {SuperGPQA_test_max_length} , kina : {kina_max_length}")

In [ ]:
SuperGPQA_train[0]
"""
Prompt <- SuperGPQA_train[i]['question'] + ",".join(SuperGPQA_train[i]['options']) + SuperGPQA_train[i]['reference']
"""
SuperGPQA_train[0]['question'] + ",".join(SuperGPQA_train[0]['options']) + SuperGPQA_train[0]['reference']

SuperGPQA_train = SuperGPQA_train.map(data_process.build_mcqa_prompt)
SuperGPQA_train[0]

In [ ]:
kina[0].keys()
kina[0]

# how to approach to kina
def build_kina_prompt(row):
    options_text = "\n".join(
        f"{option['key']}. {option['answer'].strip()}"
        for option in row["options"]
    )

    prompt = (
        f"Question:\n"
        f"{row['question'].strip()}\n\n"
        f"The answer choices are as follows:\n"
        f"{options_text}\n\n"
        f"Select the correct answer and respond with only its letter."
    )

    return {
        "prompt": prompt
    }


kina = kina.map(build_kina_prompt)

packing vs padding. generally, packing is used in pretraining with long text not sft/pre training with mcqa, etc

if using mcqa in sft then only masking except answer part with -100 token.

overlap is needed to train with context in pre trainig with long text.

# **processing tokenizer**

In [ ]:
import re
import time
from typing import List , Dict


SPECIAL_TOKENS = [
    "<pad>",
    "<bos>",
    "<eos>",
    "<unk>",
    "<endoftext>",
    "<question>",
    "<choices>",
    "<answer>",
]

class Tokenizer:
  def __init__(self , data_name : str = None , dictionary : dict[str , int] = {} , special_tokens : List[str] =SPECIAL_TOKENS):
    self.data_name = data_name
    self.dictionary = dictionary
    self.max_count = int(len(self.dictionary))-1 if self.dictionary is not None else -1
    self.special_tokens=special_tokens if special_tokens is not None else []
    self.special_token_pattern = "|".join(
    re.escape(token)
    for token in sorted(special_tokens, key=len, reverse=True)
    )

  # tokenize -> encoder
  def tokenize(self,num_samples : list[str | None] = None) -> list[str | None]:

    # need parallel since it is one by one and take long time and it's inefficient
    tokens_list = []
    for text in num_samples:

      # #, ##, ### 제거
      text = re.sub(r"#+", " ", text)

      # 영어 단어와 문장부호를 각각 추출
      raw_tokens = re.findall(
        rf"""
        {self.special_token_pattern}               # special tokens
        |\\[A-Za-z]+                          # LaTeX commands: \frac, \sqrt, \alpha
        |\n                                   # newline
        |[A-Za-z]+(?:'[A-Za-z]+)?             # English words: don't, model
        |\d+(?:\.\d+)?(?:[eE][+-]?\d+)?       # integers, decimals, scientific notation
        |\S                                   # every other non-whitespace character
        """,
        text,
        flags=re.VERBOSE,
      )

      tokens = []

      tokens.extend(raw_tokens)
      """
      for token in raw_tokens:
        match = re.fullmatch(
          r"([A-Za-z]+?)(ing|ed)",
          token,
          flags=re.IGNORECASE,
        )

        if match:
          tokens.append(match.group(1))
          tokens.append(match.group(2))
        else:
          tokens.append(token)
      """

      tokens_list.extend(tokens)
    return tokens_list

  def tokenizer_encoding(self , text_list : list[str | None]) -> Dict:
    if self.special_tokens is not None:
      for special_word in self.special_tokens:
        text_list.append(special_word)

    for sample in text_list:
      # since dict is hash table, its O(1) in searching
      if sample not in self.dictionary:
        self.max_count += 1
        self.dictionary[sample] = int(self.max_count)
    return self.dictionary

  def encode(
        self,
        text: str,
        add_bos: bool = False,
        add_eos: bool = False,
    ) -> list[int]:

        tokens = self.tokenize([text])

        if "<unk>" not in self.dictionary:
            raise ValueError(
                "Build the vocabulary before calling encode()."
            )

        unk_id = self.dictionary["<unk>"]

        token_ids = [
            self.dictionary.get(token, unk_id)
            for token in tokens
        ]

        if add_bos:
            token_ids.insert(
                0,
                self.dictionary["<bos>"],
            )

        if add_eos:
            token_ids.append(
                self.dictionary["<eos>"],
            )

        return token_ids

  def inserting_special_token(self, special_token : str | None = None):
    assert type(special_token) == str, "Warning! special_token you added is not str."
    if isinstance(special_token , str):
      if special_token not in self.dictionary.keys():
        self.dictionary[special_token] = self.max_count + 1
        self.max_count += 1

  def print_max_num_encoder_dictionary(self):
    print(f"whole number of tokens/index of dictionary : {self.max_count}")

Might be good to mapping natural language to index with semantic criteria

In [ ]:
dictionary = {'son' : 1, 'daughter' : 2}
print(len(dictionary))
tokenizer = Tokenizer(dictionary=dictionary)
tokens=tokenizer.tokenize(['I\n want to go to school but yesterday there was ##.','I want to go to school but yesterday there was ##.'])
dic_token=tokenizer.tokenizer_encoding(tokens)
print(dic_token)
tokenizer.print_max_num_encoder_dictionary()
type(dic_token['there'])

2
{'son': 1, 'daughter': 2, 'I': 3, '\n': 4, 'want': 5, 'to': 6, 'go': 7, 'school': 8, 'but': 9, 'yesterday': 10, 'there': 11, 'was': 12, '.': 13, '<pad>': 14, '<bos>': 15, '<eos>': 16, '<unk>': 17, '<endoftext>': 18, '<question>': 19, '<choices>': 20, '<answer>': 21}
whole number of tokens/index of dictionary : 21


int

# **torch dataset bulding**

In [ ]:
import math
import torch
from torch.utils.data import Dataset as TorchDataset


class Train_Dataset(TorchDataset):
    def __init__(
        self,
        dataset,
        tokenizer,
        context_length: int,
        text_column: str = "prompt",
    ):
        if context_length <= 0:
            raise ValueError("context_length must be positive.")

        if text_column not in dataset.column_names:
            raise ValueError(
                f"{text_column!r} is not in dataset.column_names."
            )

        self.context_length = context_length

        all_token_ids: list[int] = []

        for text in dataset[text_column]:
            if not isinstance(text, str):
                text = str(text)

            token_ids = tokenizer.encode(
                text,
                add_bos=True,
                add_eos=True,
            )

            all_token_ids.extend(token_ids)

        if len(all_token_ids) < 1:
            raise ValueError(
                "At least one token is required."
            )

        self.token_ids = torch.tensor(
            all_token_ids,
            dtype=torch.long,
        )

        # include remain intervals to sample , match.ceil : 반올림함수
        self.num_sequences = math.ceil(
            (len(self.token_ids) - 1)
            / self.context_length
        )

    def __len__(self) -> int:
        return self.num_sequences

    def __getitem__(
        self,
        index: int,
    ) -> dict[str, torch.Tensor]:

        if index < 0 or index >= self.num_sequences:
            raise IndexError("Dataset index out of range.")

        start = index * self.context_length

        end = min(
            start + self.context_length + 1,
            len(self.token_ids),
        )

        tokens = self.token_ids[start:end]

        return {
            "input_ids": tokens[:-1],
            "labels": tokens[1:],
        }

In [ ]:
context_length = 256

train_dataset = Train_Dataset(
    dataset=train_prompt_dataset,
    tokenizer=tokenizer,
    context_length=context_length,
    text_column="prompt",
)

In [ ]:
sample = train_dataset[0]

print(sample["input_ids"].shape)
print(sample["labels"].shape)

In [ ]:
print(
    torch.equal(
        sample["input_ids"][1:],
        sample["labels"][:-1],
    )
)

In [ ]:
from functools import partial
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence


def collate_function(
    batch,
    pad_token_id: int,
):
    input_ids = pad_sequence(
        [sample["input_ids"] for sample in batch],
        batch_first=True,
        padding_value=pad_token_id,
    )

    labels = pad_sequence(
        [sample["labels"] for sample in batch],
        batch_first=True,
        padding_value=-100,
    )

    attention_mask = (
        input_ids != pad_token_id
    ).long() # .long() makes torch.bool -> torch.int 0/1

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": attention_mask,
    }


train_loader = DataLoader(
    train_dataset,
    batch_size=config['batch_num'],
    shuffle=True,
    drop_last=False,
    collate_fn=partial(
        collate_function,
        pad_token_id=tokenizer.dictionary["<pad>"],
    ),
)

In [ ]:
batch = next(iter(train_loader))

print(batch["input_ids"].shape)
print(batch["labels"].shape)

# **processing positional embedding**

q, k: [batch_size, num_heads, sequence_length, head_dim]

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision , torchaudio

class RoPE_Positional_Embedding(nn.Module):
  def __init__(self , batch_num, embed_dim, context_length , head_dim):
    super().__init__()
    self.batch_num = batch_num
    self.embed_dim = embed_dim
    self.context_length = context_length
    self.head_dim = head_dim

    if head_dim % 2 != 0:
      raise ValueError("head_dim should be even")

    if embed_dim % head_dim != 0:
      raise ValueError("embed_dim should be divided by head_dim")

    p = torch.arange(0, context_length , 1, dtype=torch.float32)
    p=p[None,None,:,None]

    i=torch.arange(0, head_dim , 2, dtype = torch.float32)

    tmp_frequency = 10000 ** (((-1) * i) / head_dim)

    tmp_frequency=tmp_frequency[None,None,None,:]

    # frequency *=p , frequency might not change its shape so it could make error. need to use new variable.
    frequency = (tmp_frequency * p).repeat(int(batch_num), int(embed_dim//head_dim),int(1),int(1))

    self.register_buffer(
      "frequency",
      frequency,
      persistent=False,
    )

  def transformation(self):
    return torch.sin(self.frequency) , torch.cos(self.frequency)

  def construct_embedding(self,x, sin, cos):
    x_even=x[...,0::2]
    x_odd=x[...,1::2]

    rotate_even=x_even * cos - x_odd * sin
    rotate_odd=x_even * cos + x_odd * sin

    return torch.stack([rotate_even,rotate_odd],dim=-1).flatten(-2)

In [ ]:
import torch
batch_num=4
embed_dim=32
head_dim = 16
context_length = 4

"""
p = torch.arange(0, context_length , 1, dtype=torch.float32)
p=p[None,None,:,None]

i=torch.arange(0, head_dim , 2, dtype = torch.float32)

frequency = 10000 ** (((-1) * i) / head_dim)
frequency=frequency[None,None,None,:]
new=p*frequency
"""

positional_embedding=RoPE_Positional_Embedding(batch_num,embed_dim,context_length,head_dim)
positional_embedding_0, positional_embedding_1 = positional_embedding.transformation()
print(positional_embedding_0.shape , positional_embedding_1.shape)

torch.Size([4, 2, 4, 8]) torch.Size([4, 2, 4, 8])


# **Constructing Attention**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision, torchaudio

# **self.attention block**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision, torchaudio

class Self_Attention(nn.Module):
  """
  If need causal masking -> do .causal_masking(attention_matrix) <- method

  """
  def __init__(self, batch_num , embed_dim, context_length, num_head , position_flag : int , causal_process_flag : bool):
    super().__init__()
    self.batch_num = batch_num
    self.embed_dim = embed_dim
    self.context_length=context_length
    self.num_head = num_head
    self.head_dim = embed_dim // num_head
    self.position_flag = position_flag # 0 : RoPE, 1 : normal positional embedding
    self.causal_process_flag = causal_process_flag # if causal then True

    if embed_dim % num_head != 0:
      raise ValueError("embed_dim % num_head should be zero")

    self.positional_embedding = RoPE_Positional_Embedding(batch_num, embed_dim, context_length , embed_dim//num_head)

    self.query_embedding =nn.Parameter(torch.randn(self.embed_dim, self.embed_dim)) # Normal distribution
    self.key_embedding = nn.Parameter(torch.randn(self.embed_dim, self.embed_dim))
    self.value_embedding = nn.Parameter(torch.randn(self.embed_dim, self.embed_dim))

  def construct_q_k_v(self, input):
    return input@self.query_embedding ,input@self.key_embedding,input@self.value_embedding

  def causal_masking(self, attention_matrix):

    if attention_matrix.shape != torch.Size([self.batch_num, self.num_head, self.context_length, self.context_length]):
      raise ValueError("attention_matrix.shape is wrong. Should fix this!")
    # attention_matrix.shape : [B,H,C,C] before multiplying value embedding
    mask =torch.triu(
        torch.ones(
            self.context_length , self.context_length, \
            device = attention_matrix.device,
            dtype = torch.bool),
        diagonal=1
        )

    return attention_matrix.masked_fill(mask[None,None,:,:],-torch.inf)

  # not constructed
  def normal_positional_attention(self, query, key, value):
    """
    torch.sqrt(torch.tensor(self.head_dim)) is made in cpu tensor. error might occur.
    """
    pre_attention_weight = (query @ key.transpose(-1,-2)) / torch.sqrt(torch.tensor(self.head_dim))
    attention_score=(torch.softmax(pre_attention_weight , dim=-1)) @ value
    return attention_score

  def rope_positional_attention(self, query, key, value , causal_process_flag : bool):
    sin_transformation,cos_transformation=self.positional_embedding.transformation()
    query=self.positional_embedding.construct_embedding(query,sin_transformation,cos_transformation)
    key=self.positional_embedding.construct_embedding(key,sin_transformation,cos_transformation)

    pre_attention_weight = (query @ key.transpose(-1,-2)) / (self.head_dim**0.5)

    if causal_process_flag:
      causal_pre_attention_weight=self.causal_masking(pre_attention_weight)
      return (torch.softmax(causal_pre_attention_weight , dim=-1)) @ value

    return torch.softmax(pre_attention_weight , dim=-1) @ value

  # construct_q_k_v -> attention -> forward
  def forward(self , x): # x is just input

    if x.shape != torch.Size([self.batch_num, self.context_length, self.embed_dim]):
      raise ValueError("input data's shape is not what this model wants")

    if self.position_flag == 0:
      query,key,value=self.construct_q_k_v(x)
      query=query.view(self.batch_num, self.context_length , self.num_head, self.embed_dim//self.num_head).transpose(1,2).contiguous()
      key=key.view(self.batch_num, self.context_length , self.num_head, self.embed_dim//self.num_head).transpose(1,2).contiguous()
      value=value.view(self.batch_num, self.context_length , self.num_head, self.embed_dim//self.num_head).transpose(1,2).contiguous()

      if query.shape != torch.Size([self.batch_num, self.num_head, self.context_length, self.head_dim]) or \
        key.shape != torch.Size([self.batch_num, self.num_head, self.context_length, self.head_dim]) or \
        value.shape != torch.Size([self.batch_num, self.num_head, self.context_length, self.head_dim]):
        raise ValueError("query.shape and key.shape and value.shape must be equal to [batch_num,num_head, context_length , head_dim]")

      attention_score=self.rope_positional_attention(query,key,value,self.causal_process_flag)
    else:
      raise ValueError("normal positional embedding is not yet built.")

    assert self.position_flag == 0 or 1, "Warning! self.position_flag should be either 0/1."

    return attention_score.transpose(2,1).reshape(self.batch_num,self.context_length,self.embed_dim)

In [ ]:
mask=torch.triu(torch.ones(self.batch_num , self.num_head, self.context_length , self.context_length, \
                device = #class_name#.device,
                dtype = torch.bool)

                , diagonal=1)

mask=mask.masked_fill(
    mask,
    -torch.inf
)


SyntaxError: invalid syntax (2466697702.py, line 3)

In [ ]:
batch_num = 2
embed_dim = 8
context_length= 16
num_head= 4
position_flag= 0
causal_process_flag= False
x = torch.rand(batch_num, context_length, embed_dim)

self_attention=Self_Attention(batch_num, embed_dim, context_length, num_head, position_flag , causal_process_flag)

print(f"x : {x.shape}\n", sep="")
print(f"attention : {self_attention(x).shape}")

x : torch.Size([2, 16, 8])

attention : torch.Size([2, 16, 8])


In [ ]:
torch.tensor([[1,2],[3,4]]).shape == torch.tensor([2,2])

False

# **transformer block building**

In [ ]:
class Transformer(nn.Module):

  def __init__(self, batch_num : int, embed_dim : int, context_length : int, num_head : int, position_flag : int, causal_process_flag : bool,vocab_dim : int, crossentropy_flag : bool , is_transformer_block_is_final : bool):
    super().__init__()
    self.batch_num=batch_num
    self.embed_dim=embed_dim
    self.context_length=context_length
    self.num_head=num_head
    self.position_flag=position_flag
    self.causal_process_flag=causal_process_flag
    self.is_transformer_block_is_final = is_transformer_block_is_final

    if self.is_transformer_block_is_final:
      self.proj = nn.Linear(embed_dim, vocab_dim)
      self.softmax = nn.Softmax(dim=-1)
      self.layernorm=nn.LayerNorm(embed_dim)

    self.feed_forward = nn.Sequential(
    nn.Linear(embed_dim, embed_dim * 4),
    nn.GELU(),
    nn.Linear(embed_dim * 4, embed_dim),
    nn.Dropout(0.1),
    )

    self.layernorm_list=nn.ModuleList([nn.LayerNorm(embed_dim) for _ in range(2)])

    self.crossentropy_flag = crossentropy_flag

    self.causal_attention=Self_Attention(batch_num, embed_dim , context_length, num_head, position_flag, 1) # if causal_process_flag==1, it is causal_attention_process


  def forward(self, x):

    normalized_x=self.layernorm_list[0](x)
    z=self.causal_attention(normalized_x)
    z+=x
    pre_z=z
    z=self.layernorm_list[1](z)
    z=self.feed_forward(z)+pre_z

    if self.is_transformer_block_is_final:
      z=self.layernorm(z)
      logit=self.proj(z)
      vocab_prob=self.softmax(logit)
      return vocab_prob if not self.crossentropy_flag else logit
    else:
      return z

# **stacking transformer block **

In [ ]:
batch_num=2
embed_dim=32
context_length=16
num_head=4
position_flag=0
causal_process_flag=True
vocab_dim=128
crossentropy_flag=0
is_transformer_block_is_final=0

class Stacking(nn.Module):

  def __init__(self, block_num : int, batch_num : int, embed_dim : int, context_length : int, num_head:int, position_flag, causal_process_flag, vocab_dim, crossentropy_flag,is_transformer_block_is_final):
    super().__init__()
    self.block_num = block_num
    self.batch_num=batch_num
    self.embed_dim=embed_dim
    self.context_length=context_length
    self.num_head=num_head
    self.position_flag=position_flag
    self.causal_process_flag=causal_process_flag
    self.vocab_dim=vocab_dim
    self.crossentropy_flag=crossentropy_flag
    self.is_transformer_block_is_final=is_transformer_block_is_final

    self.transformer_block_list = nn.ModuleList([Transformer(batch_num, \
            embed_dim, \
            context_length, \
            num_head, \
            position_flag, \
            causal_process_flag, \
            vocab_dim, \
            crossentropy_flag, \
            is_transformer_block_is_final=False
            ) for _ in range(block_num-1)])

    self.final_block = Transformer(batch_num, \
            embed_dim, \
            context_length, \
            num_head, \
            position_flag, \
            causal_process_flag, \
            vocab_dim, \
            crossentropy_flag, \
            is_transformer_block_is_final=True
            )
  def forward(self, x):
    for transformer_block in self.transformer_block_list:
      x=transformer_block(x)
    return self.final_block(x)

# **train pipeline building**

Loss **Constructing**

In [ ]:
"""
In Training Compute-Optimal Large Language Models by DeepMind,
In page 6, In Model fitting, to estimate, they use Hyber Loss between predicted and observed log loss using L-BFGS algorithm.

`L(N , D) = E + A/N^alpha + B/D^beta, (A,B,E,alpha, beta)`
"""

import torch.nn.functional as F

def huber_loss_with_log(self, prediction : torch.tensor, label : torch.tensor, epsilon=1e-12):
  pred=torch.log(prediction + epsilon)
  answer = torch.log(label + epsilon)

  return F.huber_loss(
    pred,
    answer,
    delta=1e-3,
    reduction="sum",
    )

"""
more loss required
"""

train **argu**

In [ ]:

config = {
    'output_dir' : ,
    'learning_rate' : 2e-5,
    'per_device_train_batch_size' : 16,
    'per_device_eval_batch_size' : 16,
    'num_train_epoch' : 2,
    'weight_decay' : ,    # scheduling strategy is from deepmind paper.
    'eval_strategy' : 'epoch',
    'save_strategy' : 'epoch',
    'push_to_hub' : True,
    'load_checkpoint' : False,
    'load_specific_checkpoint' : False,
    'block_num' : 8, # number of transformer
    'batch_num' : 2,
    'embed_dim' : 512,
    'context_length' : 256,
    'num_head' : 8,
    'position_flag' : 0,  # 0 : RoPE , 1: normal positional embedding
    'causal_process_flag' : True, # decoder only version
    'vocab_dim': 16384,
    'crossentropy_flag': False, # not using cross entropy
    'is_transformer_block_is_final': False, # Whether or not to use projection and softmax to output z

}

In [ ]:
!pip install -U huggingface_hub accelerate datasets trl bitsandbytes

need huggingface token, and login using hub / enroll in os.enviorn

In [ ]:
from transformers import Trainer TrainingArguments
import accelerate # for distributed environment

training_args = TrainingArguments(
    output_dir=config['output_dir'],
    learning_rate=config['learning_rate'],
    per_device_train_batch_size=config['per_device_train_batch_size'],
    per_device_eval_batch_size=config['per_device_eval_batch_size'],
    num_train_epochs=config['num_train_epochs'],
    weight_decay=config['weight_decay'],        # scheduling strategy is from deepmind paper.
    eval_strategy=config['eval_strategy'],
    save_strategy=config['save_strategy'],
    load_best_model_at_end=True,
    push_to_hub=config['push_to_hub'],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

if config['load_checkpoint'] == False:
  trainer.train()
else:
  # 최신 체크포인트에서 재개 restart from recent checkpoint
  if config['load_specific_checkpoint'] == True:
    # 출력 디렉토리에 저장된 특정 체크포인트에서 재개 restart from specific checkpoint from output directory
    trainer.train(resume_from_checkpoint="your-model/checkpoint-1000")
  else:
    trainer.train(resume_from_checkpoint=True)

# **Final Model**

In [ ]:
class DecoderLanguageModel(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        block_num: int,
        embed_dim: int,
        context_length: int,
        num_head: int,
    ):
        super().__init__()

        self.vocab_size = vocab_size

        self.token_embedding = nn.Embedding(
            vocab_size,
            embed_dim,
        )

        self.blocks = Stacking(
            block_num=block_num,
            embed_dim=embed_dim,
            context_length=context_length,
            num_head=num_head,
            vocab_dim=vocab_size,
        )

    def forward(self, input_ids):
        # [B, T] → [B, T, D]
        x = self.token_embedding(input_ids)

        # [B, T, D] → [B, T, V]
        logits = self.blocks(x)

        return logits